# 08 tuning — XGBoost with fold-safe threshold selection

Fork of `08_xgboost.ipynb`. Everything is identical except that class thresholds are optimized per fold on a training-part holdout instead of on the pooled out-of-fold predictions, so `mean_val_qwk` is not inflated by validation-fold leakage. Results are saved to `results/xgboost_tuning_cv_results.csv`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

_ROOT = Path.cwd()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from scipy.optimize import minimize
from sklearn.base import clone
from sklearn.linear_model import Ridge
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier, XGBRegressor

from src.config import ID_COLUMN, PROCESSED_DIR, PROJECT_ROOT, RANDOM_STATE, RESULTS_DIR, TARGET
from src.evaluation import create_cv_splits, evaluate_model, quadratic_weighted_kappa
from src.imputation import make_preprocessor

experiment_results: list[dict] = []
train_features = pd.read_parquet(PROCESSED_DIR / "train_features.parquet")
print("Train features:", train_features.shape)

## Prepare data and reusable QWK helpers

In [ ]:
labeled = train_features[train_features[TARGET].notna()].reset_index(drop=True)
feature_columns = [c for c in labeled.columns if c not in {ID_COLUMN, TARGET}]
X = labeled[feature_columns].copy()
y = labeled[TARGET].astype(int)
ids = labeled[ID_COLUMN]
cv_splits = create_cv_splits(X=X, y=y, ids=ids)
print("X:", X.shape, "| class counts:", y.value_counts().sort_index().to_dict())

def predictions_to_classes(predictions, thresholds):
    return np.digitize(np.asarray(predictions), np.sort(thresholds))

def _negative_qwk(thresholds, predictions, y_true):
    return -cohen_kappa_score(y_true, predictions_to_classes(predictions, thresholds), weights="quadratic")

def optimize_thresholds(predictions, y_true, initial=(0.5, 1.5, 2.5)):
    result = minimize(
        _negative_qwk, np.asarray(initial, dtype=float),
        args=(np.asarray(predictions), np.asarray(y_true)), method="Nelder-Mead",
    )
    return np.sort(result.x)

## XGBoost classifier baseline

The baseline deliberately treats `sii` as four unordered classes. Median imputation and one-hot encoding are learned independently inside each outer fold. This provides the direct comparison that motivates ordinal regression.

In [ ]:
classifier_pipeline = Pipeline([
    ("preprocessor", make_preprocessor(X, strategy="median", scale=False)),
    ("model", XGBClassifier(
        objective="multi:softprob", num_class=4, eval_metric="mlogloss",
        n_estimators=500, random_state=RANDOM_STATE, n_jobs=-1,
    )),
])
result_classifier = evaluate_model(classifier_pipeline, X, y, cv_splits)
experiment_results.append({
    "setup": "XGBClassifier | Layer A, default-style baseline",
    "mean_val_qwk": np.mean(result_classifier["validation_scores"]),
    "val_std_qwk": np.std(result_classifier["validation_scores"]),
    "mean_train_qwk": np.mean(result_classifier["training_scores"]),
})

## Regression + QWK-optimized thresholds

Regression respects the order of the target. For fold-safe early stopping, every outer training fold is split again into a fit portion and a stopping portion. The preprocessor is fit only on the fit portion; neither the stopping data nor the outer validation data influence its statistics.

Thresholds are optimized per fold on that fold's stopping holdout only. The outer validation fold is scored with those thresholds but never used to choose them.

In [ ]:
def fit_predict_oof_regressor(
    build_model, X_frame, y, cv_splits, sample_weight=None,
    early_stopping_fraction=0.15, verbose=True, preprocessor_factory=None,
):
    """Fold-safe OOF predictions with fold-local thresholds.

    `preprocessor_factory(frame) -> unfitted transformer` defaults to median
    imputation (the notebook's original behavior); pass a different factory
    to compare preprocessing choices while keeping the model fixed.
    """
    if preprocessor_factory is None:
        preprocessor_factory = lambda frame: make_preprocessor(frame, strategy="median", scale=False)
    oof = np.zeros(len(y), dtype=float)
    oof_classes = np.zeros(len(y), dtype=int)
    fold_scores = []
    fold_thresholds = []
    for fold, (train_idx, val_idx) in enumerate(cv_splits, start=1):
        X_outer, y_outer = X_frame.iloc[train_idx], y.iloc[train_idx]
        fit_pos, stop_pos = train_test_split(
            np.arange(len(X_outer)), test_size=early_stopping_fraction,
            random_state=RANDOM_STATE, stratify=y_outer,
        )
        preprocessor = preprocessor_factory(X_outer.iloc[fit_pos])
        X_fit = preprocessor.fit_transform(X_outer.iloc[fit_pos])
        X_stop = preprocessor.transform(X_outer.iloc[stop_pos])
        X_val = preprocessor.transform(X_frame.iloc[val_idx])
        model = build_model()
        fit_kwargs = {"eval_set": [(X_stop, y_outer.iloc[stop_pos])], "verbose": False}
        if sample_weight is not None:
            fit_kwargs["sample_weight"] = np.asarray(sample_weight)[train_idx][fit_pos]
        model.fit(X_fit, y_outer.iloc[fit_pos], **fit_kwargs)

        # Thresholds come from the stopping holdout only
        stop_predictions = model.predict(X_stop)
        thresholds = optimize_thresholds(stop_predictions, y_outer.iloc[stop_pos])
        fold_thresholds.append(thresholds)

        val_predictions = model.predict(X_val)
        oof[val_idx] = val_predictions
        oof_classes[val_idx] = predictions_to_classes(val_predictions, thresholds)

        fold_score = quadratic_weighted_kappa(y.iloc[val_idx], oof_classes[val_idx])
        fold_scores.append(fold_score)
        if verbose:
            print(
                f"Fold {fold}: validation QWK={fold_score:.4f} "
                f"(thresholds={thresholds.round(3)}), best iteration={model.best_iteration}"
            )
    return oof, oof_classes, fold_scores, fold_thresholds

def build_default_regressor():
    return XGBRegressor(
        objective="reg:squarederror", eval_metric="rmse", n_estimators=3000,
        learning_rate=0.03, max_depth=6, early_stopping_rounds=100,
        random_state=RANDOM_STATE, n_jobs=-1, tree_method="hist",
    )

In [ ]:
oof_default, oof_classes_default, default_fold_scores, thresholds_default = fit_predict_oof_regressor(
    build_default_regressor, X, y, cv_splits
)
fixed_qwk = quadratic_weighted_kappa(y, predictions_to_classes(oof_default, [0.5, 1.5, 2.5]))
default_qwk = quadratic_weighted_kappa(y, oof_classes_default)
print("Fixed thresholds QWK:", round(fixed_qwk, 4))
print("Per-fold thresholds (fit on each fold's training-part holdout):", [t.round(3).tolist() for t in thresholds_default])
print("Fold-local optimized-threshold QWK:", round(default_qwk, 4))
experiment_results.append({
    "setup": "XGBRegressor | Layer A, default params + optimized thresholds",
    "mean_val_qwk": default_qwk, "val_std_qwk": np.std(default_fold_scores), "mean_train_qwk": np.nan,
})

## Class imbalance

Test inverse-square-root class weights on the regressor. QWK already penalizes distant ordinal mistakes, so weighting is measured rather than assumed to help.

In [ ]:
class_counts = y.value_counts()
class_weight_map = (1.0 / np.sqrt(class_counts)).to_dict()
sample_weight = y.map(class_weight_map).to_numpy()
sample_weight = sample_weight / sample_weight.mean()
oof_weighted, oof_classes_weighted, weighted_fold_scores, thresholds_weighted = fit_predict_oof_regressor(
    build_default_regressor, X, y, cv_splits, sample_weight=sample_weight,
)
weighted_qwk = quadratic_weighted_kappa(y, oof_classes_weighted)
print("Weighted regressor QWK:", round(weighted_qwk, 4))
experiment_results.append({
    "setup": "XGBRegressor | class-weighted + optimized thresholds",
    "mean_val_qwk": weighted_qwk, "val_std_qwk": np.std(weighted_fold_scores), "mean_train_qwk": np.nan,
})

## Missing-value handling: median imputation vs. native NaN

`tree_method="hist"` learns an optimal default split direction for missing values on its own, so imputing numeric columns before fitting is not required for XGBoost the way it is for a linear model. This compares the two options with everything else fixed: the same `build_default_regressor` config, the same features, and the same categorical encoding (most-frequent impute + one-hot) — only the numeric branch changes, median fill vs. passthrough with NaN kept. Thresholds stay fold-local as everywhere else in this notebook.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder


def make_native_nan_preprocessor(feature_frame):
    """Numeric columns pass through with NaN kept; categorical columns are
    imputed (most-frequent) and one-hot encoded, exactly as in make_preprocessor."""
    categorical_columns = feature_frame.select_dtypes(include=["object", "string", "category"]).columns.tolist()
    numeric_columns = feature_frame.select_dtypes(include="number").columns.tolist()
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ])
    return ColumnTransformer(
        transformers=[
            ("numeric", "passthrough", numeric_columns),
            ("categorical", categorical_pipeline, categorical_columns),
        ],
        remainder="drop",
    )


median_preprocessor_factory = lambda frame: make_preprocessor(frame, strategy="median", scale=False)

oof_native, oof_classes_native, native_fold_scores, thresholds_native = fit_predict_oof_regressor(
    build_default_regressor, X, y, cv_splits, preprocessor_factory=make_native_nan_preprocessor,
)
native_qwk = quadratic_weighted_kappa(y, oof_classes_native)
print("\nNative-NaN regressor QWK:", round(native_qwk, 4))

missing_value_comparison = pd.DataFrame([
    {"preprocessing": "median imputation", "mean_val_qwk": default_qwk, "val_std_qwk": np.std(default_fold_scores)},
    {"preprocessing": "native NaN (no numeric imputation)", "mean_val_qwk": native_qwk, "val_std_qwk": np.std(native_fold_scores)},
]).round(4).sort_values("mean_val_qwk", ascending=False)

missing_value_comparison_path = RESULTS_DIR / "xgboost_missing_value_comparison.csv"
missing_value_comparison.to_csv(missing_value_comparison_path, index=False)
print("Saved results to:", missing_value_comparison_path)

best_preprocessing_name = missing_value_comparison.iloc[0]["preprocessing"]
best_preprocessor_factory = (
    make_native_nan_preprocessor if best_preprocessing_name.startswith("native")
    else median_preprocessor_factory
)
print("Chosen preprocessing for hyperparameter tuning below:", best_preprocessing_name)
missing_value_comparison

**Conclusion:** median imputation wins — **0.3944** vs. **0.3791** for native NaN, both at the same fixed `build_default_regressor` config. The gap (std 0.0185 vs. 0.0439) suggests native NaN is also less stable across folds here, though `tree_method="hist"` is usually competitive with imputation in the literature — this dataset's specific missingness pattern (block-structured, e.g. whole BIA/Physical panels missing together) may be easier for an explicit median fill to exploit than for per-split default-direction learning. Median imputation is used for the hyperparameter search below.

## Optuna hyperparameter tuning

Tune tree depth, `min_child_weight`, regularization (`reg_alpha`/`reg_lambda`), and the row/column subsampling fractions against OOF QWK, using the missing-value handling chosen above and fold-local early stopping. Forty trials matches notebook 07; increase the budget after the pipeline has been validated end to end.

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.12, log=True),
        "max_depth": trial.suggest_int("max_depth", 2, 8),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 20.0, log=True),
        "subsample": trial.suggest_float("subsample", 0.55, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.55, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 30.0, log=True),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
    }
    def build():
        return XGBRegressor(
            objective="reg:squarederror", eval_metric="rmse", n_estimators=3000,
            early_stopping_rounds=100, random_state=RANDOM_STATE, n_jobs=-1,
            tree_method="hist", **params,
        )
    _, oof_classes, _, _ = fit_predict_oof_regressor(
        build, X, y, cv_splits, preprocessor_factory=best_preprocessor_factory, verbose=False,
    )
    return quadratic_weighted_kappa(y, oof_classes)

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.optimize(objective, n_trials=40, show_progress_bar=True)
print("Best tuning QWK:", round(study.best_value, 4))
print("Best params:", study.best_params)

In [ ]:
def build_tuned_regressor():
    return XGBRegressor(
        objective="reg:squarederror", eval_metric="rmse", n_estimators=3000,
        early_stopping_rounds=100, random_state=RANDOM_STATE, n_jobs=-1,
        tree_method="hist", **study.best_params,
    )

oof_tuned, oof_classes_tuned, tuned_fold_scores, thresholds_tuned = fit_predict_oof_regressor(
    build_tuned_regressor, X, y, cv_splits, preprocessor_factory=best_preprocessor_factory,
)
tuned_qwk = quadratic_weighted_kappa(y, oof_classes_tuned)
print("Tuned QWK:", round(tuned_qwk, 4))
print("Per-fold thresholds:", [t.round(3).tolist() for t in thresholds_tuned])
experiment_results.append({
    "setup": f"XGBRegressor | Optuna-tuned ({best_preprocessing_name}) + optimized thresholds",
    "mean_val_qwk": tuned_qwk, "val_std_qwk": np.std(tuned_fold_scores), "mean_train_qwk": np.nan,
})

## Blend tuned XGBoost with Ridge

In [ ]:
# Blending needs its own thresholds per candidate weight, so each fold's Ridge and
# XGBoost models are refit here, keeping the same fit/stop split XGBoost
# already uses for early stopping. Thresholds for every weight are optimized
# on the stopping holdout (inside the training fold) and only ever scored on the untouched outer validation fold.
# Ridge always needs imputed, scaled features (it can't handle NaN), so it keeps
# median imputation regardless of which preprocessing won the comparison above;
# XGBoost uses best_preprocessor_factory to match how it was tuned.
ridge_stop_predictions, ridge_val_predictions = [], []
xgb_stop_predictions, xgb_val_predictions = [], []
stop_targets, val_index_per_fold = [], []

for train_idx, val_idx in cv_splits:
    X_outer, y_outer = X.iloc[train_idx], y.iloc[train_idx]
    fit_pos, stop_pos = train_test_split(
        np.arange(len(X_outer)), test_size=0.15,
        random_state=RANDOM_STATE, stratify=y_outer,
    )

    ridge_preprocessor = make_preprocessor(X_outer.iloc[fit_pos], strategy="median")
    X_fit_ridge = ridge_preprocessor.fit_transform(X_outer.iloc[fit_pos])
    ridge = Ridge(alpha=1.0).fit(X_fit_ridge, y_outer.iloc[fit_pos])
    ridge_stop_predictions.append(ridge.predict(ridge_preprocessor.transform(X_outer.iloc[stop_pos])))
    ridge_val_predictions.append(ridge.predict(ridge_preprocessor.transform(X.iloc[val_idx])))

    xgb_preprocessor = best_preprocessor_factory(X_outer.iloc[fit_pos])
    X_fit_xgb = xgb_preprocessor.fit_transform(X_outer.iloc[fit_pos])
    X_stop_xgb = xgb_preprocessor.transform(X_outer.iloc[stop_pos])
    xgb_model = build_tuned_regressor()
    xgb_model.fit(
        X_fit_xgb, y_outer.iloc[fit_pos],
        eval_set=[(X_stop_xgb, y_outer.iloc[stop_pos])], verbose=False,
    )
    xgb_stop_predictions.append(xgb_model.predict(X_stop_xgb))
    xgb_val_predictions.append(xgb_model.predict(xgb_preprocessor.transform(X.iloc[val_idx])))

    stop_targets.append(y_outer.iloc[stop_pos].to_numpy())
    val_index_per_fold.append(val_idx)


def blend_qwk(xgb_weight):
    """Pooled QWK for one blend weight, with thresholds fit per fold on the stop holdout."""
    oof_classes = np.zeros(len(y), dtype=int)
    for fold in range(len(cv_splits)):
        blended_stop = xgb_weight * xgb_stop_predictions[fold] + (1 - xgb_weight) * ridge_stop_predictions[fold]
        thresholds = optimize_thresholds(blended_stop, stop_targets[fold])
        blended_val = xgb_weight * xgb_val_predictions[fold] + (1 - xgb_weight) * ridge_val_predictions[fold]
        oof_classes[val_index_per_fold[fold]] = predictions_to_classes(blended_val, thresholds)
    return quadratic_weighted_kappa(y, oof_classes)


ridge_qwk = blend_qwk(0.0)
experiment_results.append({
    "setup": "Ridge | Layer A reference + optimized thresholds",
    "mean_val_qwk": ridge_qwk, "val_std_qwk": np.nan, "mean_train_qwk": np.nan,
})

best_weight, best_blend_qwk = None, -np.inf
for xgb_weight in np.arange(0.0, 1.01, 0.05):
    score = blend_qwk(xgb_weight)
    if score > best_blend_qwk:
        best_weight, best_blend_qwk = xgb_weight, score
print(f"Best blend: {best_weight:.2f} XGBoost + {1 - best_weight:.2f} Ridge -> QWK={best_blend_qwk:.4f}")
experiment_results.append({
    "setup": "Blend | tuned XGBRegressor + Ridge + optimized thresholds",
    "mean_val_qwk": best_blend_qwk, "val_std_qwk": np.nan, "mean_train_qwk": np.nan,
})

## Extra engineered features

Repeat the tuned model with the same experimental domain features used in notebook 07. They remain local to this notebook until CV demonstrates that they help.

In [ ]:
train_extra = train_features.copy()
train_extra["Fitness_Endurance_total_seconds"] = train_extra["Fitness_Endurance-Time_Mins"] * 60 + train_extra["Fitness_Endurance-Time_Sec"]
train_extra["Physical-Waist_to_Height"] = train_extra["Physical-Waist_Circumference"] / train_extra["Physical-Height"]
train_extra["Physical-Pulse_Pressure"] = train_extra["Physical-Systolic_BP"] - train_extra["Physical-Diastolic_BP"]
train_extra["FGC_total_score"] = train_extra[[
    "FGC-FGC_CU", "FGC-FGC_GSND", "FGC-FGC_GSD", "FGC-FGC_PU",
    "FGC-FGC_SRL", "FGC-FGC_SRR", "FGC-FGC_TL",
]].sum(axis=1, skipna=True)
train_extra["BMI_per_Age"] = train_extra["Physical-BMI"] / train_extra["Basic_Demos-Age"]
labeled_extra = train_extra[train_extra[TARGET].notna()].reset_index(drop=True)
X_extra = labeled_extra[[c for c in labeled_extra.columns if c not in {ID_COLUMN, TARGET}]].copy()
oof_extra, oof_classes_extra, extra_fold_scores, thresholds_extra = fit_predict_oof_regressor(
    build_tuned_regressor, X_extra, y, cv_splits, preprocessor_factory=best_preprocessor_factory,
)
extra_qwk = quadratic_weighted_kappa(y, oof_classes_extra)
print("Tuned + extra features QWK:", round(extra_qwk, 4), "| delta:", round(extra_qwk - tuned_qwk, 4))
experiment_results.append({
    "setup": "XGBRegressor | tuned + extra features + optimized thresholds",
    "mean_val_qwk": extra_qwk, "val_std_qwk": np.std(extra_fold_scores), "mean_train_qwk": np.nan,
})

## Summary and saved results

In [ ]:
summary = pd.DataFrame(experiment_results).round(4).sort_values("mean_val_qwk", ascending=False)
RESULTS_DIR.mkdir(exist_ok=True)
results_path = RESULTS_DIR / "xgboost_tuning_cv_results.csv"
summary.to_csv(results_path, index=False)
print("Saved results to:", results_path.relative_to(PROJECT_ROOT))
summary

## Reading the result

Use the sorted table rather than assuming the most complex setup wins. In particular, check whether class weighting, the Ridge blend, and extra features beat the plain tuned regressor. Thresholds throughout this notebook are already fit per fold on a training-part holdout, so `mean_val_qwk` is a fold-safe estimate; still validate the final chosen thresholds (e.g. their spread across folds) before fitting on all labeled rows and predicting the test set.

Compare this table against `results/xgboost_cv_results.csv` to see how much the pooled-threshold leak was inflating scores.